In [1]:
from evaluation.flights.evaluation import main
from datetime import date
target_date = date(2026, 6, 1)
outputs = main(test_src="model", gold_src = "sfdps", test_dates=target_date, pia_or_american_ladd_only = True, )

Loading airports from /Volumes/T2-SSD/planequery/data/interim/airports.csv
Stats for model:sfdps: {'true_positive': 243, 'false_positive': 5, 'false_negative': 6, 'precision': 0.9798387096774194, 'recall': 0.9759036144578314, 'f1': 0.9778672032193159, 'airport_ident_match_pct': 0.9135802469135802}
Stats CSV saved to /Volumes/T2-SSD/planequery/data/experiments/comparison/2026-06-26_22-03_e9c91ce_flights_evaluation.csv


In [2]:
from data_engineering.flights.flight_type import get_flights
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
df_test = get_flights(target_date)
df_gold = get_sfdps_flights_day(target_date)

In [3]:
df,stats = outputs[0]

In [4]:
COLUMNS = [
    "icao",
    "takeoff_time",
    "takeoff_airport_ident",
    "takeoff_airport_ident_distance_km",
    "landing_time",
    "landing_airport_ident",
    "landing_airport_ident_distance_km",
    "takeoff_time_df1",
    "takeoff_airport_ident_df1",
    "landing_time_df1",
    "landing_airport_ident_df1",
    "same_takeoff_airport_ident",
    "same_landing_airport_ident",
    "same_airport_ident",
    "match_status",
]
df = df.select(COLUMNS)

In [5]:
import polars as pl
df.filter(pl.col("match_status") != "both").get_column("icao").unique().to_list()

['a95016', 'a66103', 'a76d02', 'a037ee', 'a16b84']

In [12]:
icao = "a16b84"
display(df_test.filter(pl.col("icao") == icao))
display(df_gold.filter(pl.col("icao") == icao))

icao,callsign,registration,takeoff_time,takeoff_airport_ident,takeoff_airport_ident_distance_km,landing_time,landing_airport_ident,landing_airport_ident_distance_km,first_lat,first_lon,first_baro_altitude_ft,first_geom_altitude_ft,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,takeoff_airport_elevation_difference_ft,landing_airport_elevation_difference_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,f64,datetime[ms],str,f64,f64,f64,i64,i64,f64,f64,i64,i64,i64,i64,bool,bool,bool,bool,str,str,str,str
"""a16b84""","""""","""""",2026-06-01 10:03:55.790,"""KGRR""",0.233043,2026-06-01 11:28:03.180,"""KGRR""",2.243299,null,null,null,null,42.90096,-85.521829,975,975,31,181,false,true,false,false,"""""","""""","""""",""""""


icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a16b84""","""N1907M""","""N1907M""",2026-06-01 10:04:00,"""KGRR""",2026-06-01 10:41:00,"""KESC""",false,true,false,false,"""C56X""","""MEIJER DISTRIBUTION INC""","""CESSNA 560XL Citation XLS""",""""""
"""a16b84""","""N1907M""","""N1907M""",2026-06-01 10:56:00,"""KESC""",2026-06-01 11:32:00,"""KGRR""",false,true,false,false,"""C56X""","""MEIJER DISTRIBUTION INC""","""CESSNA 560XL Citation XLS""",""""""


In [13]:
from data_engineering.adsb.read_adsb import read_adsb
from evaluation.flights.visual import visualize_flight
visualize_flight(df_gold.filter(pl.col("icao") == icao), read_adsb(target_date, icaos=[icao]))